# TC-ADV 录屏版执行 Notebook

这个 Notebook 面向 ModelScope Notebook / Jupyter 场景，目标是把 `ICEWS14 录屏版` 的完整闭环固定成一套可重复执行的步骤。

默认假设：

- `TC-ADV` 位于 `/mnt/workspace/TC-ADV`
- `LMCA-TIC` 位于 `/mnt/workspace/LMCA-TIC`
- 虚拟环境位于 `/mnt/workspace/venvs/tcadv`
- 录屏配置为 `configs/experiments/icews14_record_qwen25_05b_a10.yaml`
- 本地离线小模型目录为 `/mnt/workspace/LMCA-TIC/models/Qwen2.5-0.5B-Instruct`

如果你的路径不同，只需要修改下面第一个代码单元里的变量。

In [ ]:
from pathlib import Path

ROOT_TCADV = Path('/mnt/workspace/TC-ADV')
ROOT_LMCA = Path('/mnt/workspace/LMCA-TIC')
VENV = Path('/mnt/workspace/venvs/tcadv')
CONFIG = 'configs/experiments/icews14_record_qwen25_05b_a10.yaml'
RUN_NAME = 'tcadv_icews14_record_qwen25_05b_a10'
LOCAL_MODEL = ROOT_LMCA / 'models' / 'Qwen2.5-0.5B-Instruct'
TRAIN_LOG = ROOT_TCADV / 'train_record.log'

print('ROOT_TCADV =', ROOT_TCADV)
print('ROOT_LMCA  =', ROOT_LMCA)
print('VENV       =', VENV)
print('CONFIG     =', CONFIG)
print('RUN_NAME   =', RUN_NAME)
print('LOCAL_MODEL=', LOCAL_MODEL)


## 1. 环境检查

这一步确认：

- GPU 可见
- 两个仓库路径存在
- 虚拟环境存在
- 本地模型目录存在
- 录屏配置文件存在

In [ ]:
import subprocess

cmd = f'''
set -e
nvidia-smi
echo "---- PATH CHECK ----"
test -d "{ROOT_TCADV}" && echo "TC-ADV ok: {ROOT_TCADV}"
test -d "{ROOT_LMCA}" && echo "LMCA-TIC ok: {ROOT_LMCA}"
test -d "{VENV}" && echo "venv ok: {VENV}"
test -f "{ROOT_TCADV / CONFIG}" && echo "config ok: {ROOT_TCADV / CONFIG}"
if [ -d "{LOCAL_MODEL}" ]; then
  echo "local model ok: {LOCAL_MODEL}"
else
  echo "local model missing: {LOCAL_MODEL}"
fi
'''
subprocess.run(['bash', '-lc', cmd], check=True)


## 2. 同步代码并重新安装 editable 包

如果远端工作区还有临时改动，`git pull` 前请先在终端手工执行：

```bash
git stash push -u -m "temp before sync"
```

这里的单元默认直接 `git pull`。

In [ ]:
import subprocess

cmd = f'''
set -e
source "{VENV}/bin/activate"
cd "{ROOT_TCADV}"
git pull
python -m pip install -e "{ROOT_LMCA}"
python -m pip install -e "{ROOT_TCADV}"
'''
subprocess.run(['bash', '-lc', cmd], check=True)


## 3. 清理旧输出

这一单元会删除这次录屏配置对应的旧产物，避免日志、checkpoint 和指标文件互相污染。

In [ ]:
import subprocess

cmd = f'''
set -e
cd "{ROOT_TCADV}"
rm -rf "outputs/{RUN_NAME}" \
       "logs/{RUN_NAME}" \
       "checkpoints/{RUN_NAME}" \
       "data/processed/{RUN_NAME}" \
       "{TRAIN_LOG}"
echo "cleaned old artifacts for {RUN_NAME}"
'''
subprocess.run(['bash', '-lc', cmd], check=True)


## 4. 后台启动训练

这个单元会把训练放到后台执行，并把完整输出写入：

- `/mnt/workspace/TC-ADV/train_record.log`

训练 PID 会打印出来，后续查看日志和进程状态会更方便。

In [ ]:
import subprocess

cmd = f'''
set -e
source "{VENV}/bin/activate"
cd "{ROOT_TCADV}"
nohup tc-adv train --config {CONFIG} > "{TRAIN_LOG}" 2>&1 &
echo $!
'''
result = subprocess.run(['bash', '-lc', cmd], check=True, text=True, capture_output=True)
print('training pid =', result.stdout.strip())
print('log file =', TRAIN_LOG)


## 5. 查看训练日志

这个单元会持续输出训练日志，适合录屏。

如果你还想同时展示 GPU 占用，建议在 **另一个终端** 手工执行：

```bash
watch -n 2 nvidia-smi
```

停止日志跟踪时，在 Notebook 里中断这个单元即可。

In [ ]:
import subprocess

cmd = f'''
cd "{ROOT_TCADV}"
tail -f "{TRAIN_LOG}"
'''
subprocess.run(['bash', '-lc', cmd], check=True)


## 6. 检查训练进程是否结束

如果训练已经结束，这个单元应该看不到活跃的 `tc-adv train` 进程。

In [ ]:
import subprocess

cmd = "ps -ef | grep 'tc-adv train' | grep -v grep || true"
subprocess.run(['bash', '-lc', cmd], check=True)


## 7. 显式评测 best checkpoint

虽然训练结束时通常已经产出测试结果，但这里再显式跑一次 `evaluate`，保证结果来自当前 `best` checkpoint。

In [ ]:
import subprocess

cmd = f'''
set -e
source "{VENV}/bin/activate"
cd "{ROOT_TCADV}"
tc-adv evaluate --config {CONFIG}
'''
subprocess.run(['bash', '-lc', cmd], check=True)


## 8. 读取核心结果

这里会把录屏版最重要的几个结果文件直接打印出来：

- `test_metrics.json`
- `valid_metrics.json`
- `test_diagnostics.json` 前几行
- `tc_adv.log` 最后若干行

In [ ]:
import json
from pathlib import Path

output_dir = ROOT_TCADV / 'outputs' / RUN_NAME
log_path = ROOT_TCADV / 'logs' / RUN_NAME / 'tc_adv.log'

for name in ['valid_metrics.json', 'test_metrics.json']:
    path = output_dir / name
    print(f'===== {path} =====')
    print(json.dumps(json.loads(path.read_text(encoding='utf-8')), ensure_ascii=False, indent=2))
    print()

diag_path = output_dir / 'test_diagnostics.json'
print(f'===== {diag_path} (first 2000 chars) =====')
print(diag_path.read_text(encoding='utf-8')[:2000])
print()

print(f'===== {log_path} (tail 40) =====')
lines = log_path.read_text(encoding='utf-8').splitlines()
print('\n'.join(lines[-40:]))


## 9. 可选：导出代码附录

如果你录完屏后还想顺手导出当前仓库代码清单，可以运行下面这个单元。

In [ ]:
import subprocess

cmd = f'''
set -e
source "{VENV}/bin/activate"
cd "{ROOT_TCADV}"
tc-adv export-code --output outputs/code_bundle.md
ls -lh outputs/code_bundle.md
'''
subprocess.run(['bash', '-lc', cmd], check=True)
